# Sleep-EDF Data Inspection

Google Colab notebook for inspecting Sleep-EDF sleep-cassette EEG data.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
!pip install mne -q

import os
import mne
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
cassette_path = "/content/drive/MyDrive/sleep-edf/sleep-cassette"

print("Path exists:", os.path.exists(cassette_path))

files = os.listdir(cassette_path)

print("\nNumber of files:", len(files))

print("\nFirst 20 files:")
for f in files[:20]:
    print(f)

In [ ]:
psg_files = sorted([
    os.path.join(cassette_path, f)
    for f in files
    if "PSG.edf" in f
])

hyp_files = sorted([
    os.path.join(cassette_path, f)
    for f in files
    if "Hypnogram.edf" in f
])

print("PSG count:", len(psg_files))
print("Hypnogram count:", len(hyp_files))

print("\nExample PSG:")
print(psg_files[:5])

print("\nExample Hypnogram:")
print(hyp_files[:5])

In [ ]:
idx = 0

raw = mne.io.read_raw_edf(
    psg_files[idx],
    preload=False,
    verbose=False
)

print(raw)

In [ ]:
print("\nChannel names:")
print(raw.ch_names)

print("\nSampling frequency:")
print(raw.info["sfreq"])

print("\nDuration (seconds):")
print(raw.times[-1])

print("\nDuration (hours):")
print(raw.times[-1] / 3600)

In [ ]:
eeg_channels = [ch for ch in raw.ch_names if "EEG" in ch]

print("EEG channels:")
print(eeg_channels)

raw_eeg = raw.copy().pick(eeg_channels)

print(raw_eeg)

In [ ]:
raw_eeg.load_data()

print("EEG loaded.")

In [ ]:
raw_eeg.plot(
    duration=30,
    n_channels=len(eeg_channels),
    scalings="auto"
)

In [ ]:
data = raw_eeg.get_data()

print("EEG numpy shape:")
print(data.shape)

print("\nMeaning:")
print("(num_channels, total_samples)")

In [ ]:
sfreq = raw_eeg.info["sfreq"]

channel_idx = 0

signal = data[channel_idx]

time = np.arange(len(signal)) / sfreq

plt.figure(figsize=(15,4))
plt.plot(time[:3000], signal[:3000])

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.title(
    f"Channel: {eeg_channels[channel_idx]}"
)

plt.show()

In [ ]:
annot = mne.read_annotations(hyp_files[idx])

print(annot)

print("\nFirst 20 annotations:")
print(annot[:20])

In [ ]:
raw_eeg.set_annotations(annot)

events, event_id = mne.events_from_annotations(raw_eeg)

print("\nEvent ID mapping:")
print(event_id)

print("\nEvents shape:")
print(events.shape)

print("\nFirst 10 events:")
print(events[:10])

In [ ]:
epochs = mne.make_fixed_length_epochs(
    raw_eeg,
    duration=30.0,
    preload=True,
    verbose=False
)

X = epochs.get_data()

print("Epoch data shape:")
print(X.shape)

print("\nMeaning:")
print("(num_epochs, num_channels, samples_per_epoch)")

In [ ]:
epoch_idx = 0
channel_idx = 0

epoch_signal = X[epoch_idx, channel_idx]

time = np.arange(len(epoch_signal)) / sfreq

plt.figure(figsize=(12,4))
plt.plot(time, epoch_signal)

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.title(
    f"Epoch {epoch_idx} | Channel {eeg_channels[channel_idx]}"
)

plt.show()